# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id, name and description, and fields included
print('Record Sets:')
record_sets = []
for record_set in metadata.record_sets:
    rs_id = record_set.id
    rs_name = getattr(record_set, 'name', '[no name]')
    rs_desc = getattr(record_set, 'description', '[no description]')
    print(f"- @id: {rs_id}  name: {rs_name}  description: {rs_desc}")
    # Show fields for each record set
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            field_name = getattr(field, 'name', '[no name]')
            print(f"    - Field @id: {field.id}    name: {field_name}")
    record_sets.append(rs_id)

# If only one record set, show its top records
if record_sets:
    print(f"\nFirst 3 records from record set {record_sets[0]}:")
    for idx, x in enumerate(dataset.records(record_set=record_sets[0])):
        print(x)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames
# We'll use all discovered record_set @id's
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records from record set: {record_set}")
        print(f"Columns: {df.columns.tolist()}")

# For further analysis, pick the first available record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None and main_record_set_id in dataframes:
    main_df = dataframes[main_record_set_id]
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, choose a numeric field from the main DataFrame

# Inspect columns and infer numeric columns
if main_record_set_id is not None and main_record_set_id in dataframes:
    numeric_candidates = []
    for col in main_df.columns:
        try:
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_candidates.append(col)
            else:
                converted = pd.to_numeric(main_df[col], errors='coerce')
                if converted.notnull().sum() > 0:
                    numeric_candidates.append(col)
        except Exception:
            pass
    print(f"Numeric fields detected: {numeric_candidates}")

    # We'll proceed only if there's at least one numeric field
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first numeric
        # Try to convert the column to numeric for safe calculation
        main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
        # Filter by a threshold (use median for demonstration if no domain knowledge)
        threshold = main_df[numeric_field].median() if not main_df[numeric_field].isnull().all() else 0
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (median):")
        print(filtered_df.head())

        # Normalize (z-score) the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first non-numeric field
        group_field = None
        for col in main_df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(main_df[col]):
                group_field = col
                break
        
        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No main dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the main numeric field and its normalized counterpart
if main_record_set_id is not None and main_record_set_id in dataframes and 'filtered_df' in locals():
    if numeric_candidates:
        # Histogram of the raw field
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field} (filtered)")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # Histogram of the normalized field
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel('Count')
        plt.show()

        # Boxplot grouping by group_field, if found
        if 'group_field' in locals() and group_field is not None and group_field in filtered_df.columns:
            plt.figure(figsize=(10,4))
            sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=60)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_This notebook provided an initial exploration of the FAIR² dataset 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells'. We loaded the dataset schema and records using `mlcroissant`, inspected available record sets and fields by their `@id`, extracted tabular data into pandas DataFrames, and demonstrated basic filtering, normalization, grouping, and visualization over one numeric field. For more advanced analysis, see the Croissant schema documentation or explore additional fields by their `@id`._